In [ ]:
# Debugging junyper notebook VS Code
# pip install torch transformers datasets accelerate peft bitsandbytes trl einops "python-dotenv" jupyter ipywidgets widgetsnbextension traitlets

# MACOS
%pip install torch transformers datasets accelerate peft trl pandas

In [ ]:
# Checking GPU (Windows)
!nvidia-smi

In [ ]:
# Logging into HF
import os
from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv()
token = os.getenv("HUGGINGFACE_TOKEN")
login(token=token)

In [ ]:
# Choosing model
from transformers import AutoTokenizer
model = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3.1-8B-Instruct", torch_dtype="auto", device_map="auto")
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3.1-8B-Instruct")

# Testing model by displaying vocab size
# Vocab size: # of tokens the model is designed to recognize/process
print(model.vocab_size)

In [ ]:
# Using 'plotly' instead of 'matplotlib'
# import pandas as pd
# pd.options.plotting.backend = "plotly"

# Loading dataset
from datasets import load_dataset

# Using allenai arc_easy database; Selecting the training dataset ('train') from 'dataset'
dataset = load_dataset("allenai/ai2_arc", "ARC-Easy")["train"]   # Dictionary var-type
print(dataset)

# Preprocessing function
def format_arc_easy(data):
    # Get individual fields from 'dataset'
    qid      = data["id"]
    question = data["question"]
    choices  = data["choices"]["text"]            # list[str]
    labels   = data["choices"].get("label", None) # e.g., ["A","B","C","D"] (sometimes digits)
    answer   = data.get("answerKey")              # "B" in ARC-Easy train

    if isinstance(answer, str) and len(answer) == 1:
        if labels and answer in labels:
            idx = labels.index(answer)
            label_letter = answer
        else:
            idx = ord(answer) - ord("A")
            label_letter = answer
    elif isinstance(answer, int):
        idx = answer
        if labels and 0 <= idx < len(labels):
            label_letter = labels[idx]
        else:
            label_letter = chr(ord("A") + idx)
    else:
        raise ValueError(f"Unexpected answerKey format: {answer!r}")

    # Safety check
    if not (0 <= idx < len(choices)):
        raise ValueError(f"Answer index out of range: idx={idx}, choices={choices}")

    correct_choice = choices[idx]

    # Render choices using provided labels when available; otherwise ABC...
    if labels and len(labels) == len(choices):
        rendered_choices = "\n".join(f"{lbl}. {txt}" for lbl, txt in zip(labels, choices))
    else:
        rendered_choices = "\n".join(f"{chr(ord('A')+i)}. {txt}" for i, txt in enumerate(choices))

    # Build Alpaca-style text
    instruction = (
        f"ID: {qid}\n"
        f"Question: {question}\n"
        f"Choices:\n{rendered_choices}\n"
        f"Answer: "
    )
    response = f"{label_letter}. {correct_choice}"

    return {"text": f"### Instruction:\n{instruction}\n### Response:\n{response}"}

# Apply formatting (adds a new 'text' column)
dataset = dataset.map(format_arc_easy)

# Peek at one formatted sample
print(dataset[0]["text"])

# Goal format: Instruction-Response (Alpaca-style)

# ### Instruction:
# Question: Which planet is known as the Red Planet?
# Choices:
# A. Earth
# B. Mars
# C. Venus
# Answer: 
# ### Response:
# B. Mars
